In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor

In [ ]:
FILE_NAME = "/content/passengers.xlsx"

MAX_MISSING_PCT=30

REMOVE_HIGH_MISSING = False

In [ ]:
if FILE_NAME.lower().endswith(".csv"):

    df = pd.read_csv(
        FILE_NAME
    )

elif FILE_NAME.lower().endswith(".xlsx"):

    df = pd.read_excel(
        FILE_NAME
    )

else:

    raise ValueError(
        "Please use a CSV or XLSX file."
    )


print("ORIGINAL DATA")
print(df.head())
print(
    "\nData size:",
    df.shape
)

print(
    "\nColumn names:"
)

print(
    df.columns.tolist()
)



ORIGINAL DATA
        Station      Month  Passengers
0  Adi Soemarmo 2018-01-01         NaN
1  Adi Soemarmo 2018-02-01         NaN
2  Adi Soemarmo 2018-03-01         NaN
3  Adi Soemarmo 2018-04-01         NaN
4  Adi Soemarmo 2018-05-01         NaN

Data size: (2813, 3)

Column names:
['Station', 'Month', 'Passengers']


In [ ]:
df = df.rename(columns={
 "Nama_Stasiun": "Station",
 "Bulan": "Month",
"Jumlah_Penumpang": "Passengers"
 })


required_columns = [
    "Station",
    "Month",
    "Passengers"
]


for col in required_columns:

    if col not in df.columns:

        raise ValueError(
            f"Column '{col}' was not found."
        )


# **CLEANING**

In [ ]:
df["Month"] = pd.to_datetime(
    df["Month"],
    errors="coerce")


#make all dates the first day of each month
df["Month"] = (
    df["Month"]
    .dt.to_period("M")
    .dt.to_timestamp()
)


df["Passengers"] = pd.to_numeric(
    df["Passengers"],
    errors="coerce"
)


#remove rows where Station or Month is missing
df = df.dropna(
    subset=[
        "Station",
        "Month"
    ]
).copy()


#clean station names
df["Station"] = (
    df["Station"]
    .astype(str)
    .str.strip()
)

duplicate check

If more than one record exists for the same
station and month, passenger values are summed.

This is appropriate if several records represent
components of the monthly total.

In [ ]:
duplicate_count = df.duplicated(
    subset=[
        "Station",
        "Month"
    ]
).sum()


print("DUPLICATE CHECK")

print(
    "Number of duplicate Station-Month records:",
    duplicate_count
)


df = (
    df.groupby(
        [
            "Station",
            "Month"
        ],
        as_index=False
    )
    .agg(
        Passengers=(
            "Passengers",
            lambda x: x.sum(min_count=1)
        )
    )
)


DUPLICATE CHECK
Number of duplicate Station-Month records: 0


data sorting

In [ ]:
df = df.sort_values(
    [
        "Station",
        "Month"
    ]
).reset_index(
    drop=True
)

complete monthly series
march 2019 otomatis jadi missing obs

In [ ]:
complete_data = []
for station in df["Station"].unique():

    temp = df[
        df["Station"] == station
    ].copy()

    temp = temp.sort_values(
        "Month"
    )


#Complete monthly calendar from the first to the last available month for each station

    full_months = pd.date_range(
        start=temp["Month"].min(),
        end=temp["Month"].max(),
        freq="MS"
    )
    temp = (
        temp
        .set_index("Month")
        .reindex(full_months)
    )
    temp.index.name = "Month"

    temp["Station"] = station


    complete_data.append(
        temp
    )


df_complete = pd.concat(
    complete_data
).reset_index()

df_complete = df_complete.sort_values(
    [
        "Station",
        "Month"
    ]
).reset_index(
    drop=True)

#**MISSING DATA**

Was_Missing:
0 = original passenger data available
1 = passenger value was originally missing


In [ ]:
df_complete["Was_Missing"] = (
    df_complete["Passengers"]
    .isna()
    .astype(int)
)


print("MISSING DATA")

print(
    "Total missing observations:",
    df_complete[
        "Was_Missing"
    ].sum()
)

MISSING DATA
Total missing observations: 1401


In [ ]:
missing_summary = (
    df_complete
    .groupby("Station")
    .agg(
        Total_Months=(
            "Month",
            "size"
        ),

        Missing_Months=(
            "Was_Missing",
            "sum"
        )
    )
    .reset_index()
)
missing_summary[
    "Missing_Percentage"
] = (
    missing_summary[
        "Missing_Months"
    ]
    /
    missing_summary[
        "Total_Months"
    ]

    * 100
)

missing_summary = (
    missing_summary
    .sort_values(
        "Missing_Percentage",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)

print("\nMissing data by station:")
print(missing_summary)


Missing data by station:
                             Station  Total_Months  Missing_Months  \
0                          Kebonromo            97              97   
1                      Kedungbanteng            97              96   
2                            Masaran            97              96   
3                           Klaten X            97              90   
4   Bandara Internasional Yogyakarta            97              90   
5                      Lempuyangan X            97              90   
6                       Yogyakarta X            97              90   
7                        Purwosari X            97              90   
8                              Palur            97              90   
9                     Solo Balapan X            97              88   
10                         Brambanan            97              60   
11                            Maguwo            97              59   
12                             Jenar            97              

high missing st

In [ ]:
high_missing_stations = (
    missing_summary[
        missing_summary[
            "Missing_Percentage"
        ] > MAX_MISSING_PCT
    ]["Station"]
    .tolist()
)
print(
    f"\nStations with more than "
    f"{MAX_MISSING_PCT}% missing:"
)
print(
    high_missing_stations
)

if REMOVE_HIGH_MISSING:

    df_complete = df_complete[
        ~df_complete[
            "Station"
        ].isin(
            high_missing_stations
        )
    ].copy()
    print("\nStations with high missing data "
        "have been removed.")


Stations with more than 30% missing:
['Kebonromo', 'Kedungbanteng', 'Masaran', 'Klaten X', 'Bandara Internasional Yogyakarta', 'Lempuyangan X', 'Yogyakarta X', 'Purwosari X', 'Palur', 'Solo Balapan X', 'Brambanan', 'Maguwo', 'Jenar', 'Pasarnguter', 'Sukoharjo', 'Solokota', 'Wonogiri', 'Kadipiro', 'Adi Soemarmo']


# IMPUTE MISSING VALUE
biar ga pake future leakage;
1st choice nya t-12 (pake data bulan yg sama tp di tahun sebelumnya)
2nd choice nya pake t-1

In [ ]:
def historical_imputation(group):
    group = group.sort_values(
        "Month"
    ).copy()

    values = (
        group["Passengers"]
        .astype(float)
        .to_numpy()
    )

    for i in range(
        len(values)
    ):
        if np.isnan(
            values[i]
        ):

#1st choice
            if (
                i >= 12
                and
                not np.isnan(
                    values[i - 12]
                )
            ):

                values[i] = (
                    values[i - 12]
                )

#2nd choice
            elif (
                i >= 1
                and
                not np.isnan(
                    values[i - 1]
                )
            ):
                values[i] = (
                    values[i - 1]
                )
    group[
        "Passengers_Filled"
    ] = values

    return group

df_complete = (
    df_complete
    .groupby(
        "Station",
        group_keys=False
    )
    .apply(
        historical_imputation
    )
    .reset_index(
        drop=True
    )
)

/tmp/ipykernel_1070/462690577.py:55: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


Check missing value lagi

In [ ]:
remaining_missing = (
    df_complete[
        "Passengers_Filled"
    ]
    .isna()

    .sum()
)
print("AFTER IMPUTATION")
print(
    "Remaining missing values:", remaining_missing
)


#missing val di awal bisa aja ada karna gaada prev information nya, karna gabisa di impute maka di remove
df_complete = (
    df_complete
    .dropna(
        subset=[
            "Passengers_Filled"
        ]
    )
    .copy()
)

AFTER IMPUTATION
Remaining missing values: 281


month variable (1, 2, .., 12)

In [ ]:
df_complete[
    "month_num"
] = (
    df_complete[
        "Month"
    ]
    .dt.month
)

time index (month 1 = 0, month 2 = 1, ... month 96 = 97)

In [ ]:
df_complete[
    "time_index"
] = (
    df_complete

    .groupby(
        "Station"
    )
    .cumcount()

)
print("\nExample of Time Index:")
print(df_complete[
        [
            "Station",
            "Month",
            "Passengers_Filled",
            "time_index"
        ]
    ]
    .head(15))


Example of Time Index:
         Station      Month  Passengers_Filled  time_index
23  Adi Soemarmo 2019-12-01             6925.0           0
24  Adi Soemarmo 2020-01-01            50557.0           1
25  Adi Soemarmo 2020-02-01            35131.0           2
26  Adi Soemarmo 2020-03-01             2295.0           3
27  Adi Soemarmo 2020-04-01             2295.0           4
28  Adi Soemarmo 2020-05-01             2295.0           5
29  Adi Soemarmo 2020-06-01             2295.0           6
30  Adi Soemarmo 2020-07-01             2295.0           7
31  Adi Soemarmo 2020-08-01             2295.0           8
32  Adi Soemarmo 2020-09-01             2295.0           9
33  Adi Soemarmo 2020-10-01             2295.0          10
34  Adi Soemarmo 2020-11-01             2295.0          11
35  Adi Soemarmo 2020-12-01             6925.0          12
36  Adi Soemarmo 2021-01-01              697.0          13
37  Adi Soemarmo 2021-02-01              725.0          14


#LAG

In [ ]:
lags = [
    1,
    2,
    3,
    6,
    12]

for lag in lags:
    df_complete[
        f"lag_{lag}"
    ] = (
        df_complete
        .groupby(
            "Station"
        )[
            "Passengers_Filled"
        ]
        .shift(
            lag
        )
    )

hapus rows tanpa pengaruhi suff lag history

In [ ]:
df_model = (
    df_complete
    .dropna(
        subset=[
            "lag_1",
            "lag_2",
            "lag_3",
            "lag_6",
            "lag_12"
        ]
    )

    .copy()

)
print("DATA READY FOR MODELLING")
print(
    df_model[
        [
            "Station",
            "Month",
            "Passengers_Filled",
            "month_num",
            "time_index",
            "lag_1",
            "lag_2",
            "lag_3",
            "lag_6",
            "lag_12"
        ]
    ].head()
)
print(
    "\nNumber of observations:", len(df_model))

DATA READY FOR MODELLING
         Station      Month  Passengers_Filled  month_num  time_index   lag_1  \
35  Adi Soemarmo 2020-12-01             6925.0         12          12  2295.0   
36  Adi Soemarmo 2021-01-01              697.0          1          13  6925.0   
37  Adi Soemarmo 2021-02-01              725.0          2          14   697.0   
38  Adi Soemarmo 2021-03-01             1011.0          3          15   725.0   
39  Adi Soemarmo 2021-04-01             1064.0          4          16  1011.0   

     lag_2   lag_3   lag_6   lag_12  
35  2295.0  2295.0  2295.0   6925.0  
36  2295.0  2295.0  2295.0  50557.0  
37  6925.0  2295.0  2295.0  35131.0  
38   697.0  6925.0  2295.0   2295.0  
39   725.0   697.0  2295.0   2295.0  

Number of observations: 2196


predictors & target

In [ ]:
features = [
    "Station",
    "month_num",
    "time_index",
    "lag_1",
    "lag_2",
    "lag_3",
    "lag_6",
    "lag_12"
]
target = (
    "Passengers_Filled")

# TRAIN, VALIDATION, & TEST SPLIT
###training period: sebelum 1 jan 2024
###validation period: jan - dec 2024
###test period: jan 2025 dst

In [ ]:
train = (
    df_model[
        df_model[
            "Month"
        ] < "2024-01-01"
    ]
    .copy()
)
validation = (
    df_model[
        (
            df_model[
                "Month"
            ] >= "2024-01-01"
        )
        &
        (
            df_model[
                "Month"
            ] < "2025-01-01"
        )
    ]
    .copy()
)
test = (
    df_model[
        df_model[
            "Month"
        ] >= "2025-01-01"
    ]
    .copy()
)

original target untuk training

In [ ]:
train_original = (train[train["Was_Missing"] == 0]
                  .copy())


validation_original = (validation[validation["Was_Missing"] == 0]
                       .copy())

buat X & Y data

In [ ]:
X_train = (train_original[features])
y_train = (train_original[target])
X_val = (validation_original[features])
y_val = (validation_original[target])
X_test_all = (test[features])
print("DATA SPLIT")
print(
    "Training:",
    len(train_original))
print(
    "Validation:",
    len(validation_original))
print(
    "Testing:",
    len(test))

DATA SPLIT
Training: 844
Validation: 144
Testing: 364


In [ ]:
numeric_features = [
    "month_num",
    "time_index",
    "lag_1",
    "lag_2",
    "lag_3",
    "lag_6",
    "lag_12"]
categorical_features = [
    "Station"]

#**PREPROCESSING**

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(),numeric_features),
         ("station", OneHotEncoder(handle_unknown="ignore"), categorical_features)])

#**BACKPROPAGATION (MLP MODEL)**
###input neuron nya 16, 8, output

In [ ]:
mlp = MLPRegressor(
    hidden_layer_sizes=(16,8),
    activation="relu",
    solver="adam",
    learning_rate_init=0.001,
    max_iter=2000,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=50,
    random_state=123
)

pipeline

In [ ]:
pipeline = Pipeline(
    steps=[("preprocessing",preprocessor),("mlp",mlp)])

scale target var

In [ ]:
model = TransformedTargetRegressor(
    regressor=pipeline,
    transformer=StandardScaler())

##TRAIN BPPN

In [ ]:
print("TRAINING BACKPROPAGATION MODEL")
model.fit(
    X_train,
    y_train)
print(
    "Training completed.")

TRAINING BACKPROPAGATION MODEL
Training completed.


#VALIDATION PERF

In [ ]:
if len(
    validation_original
) > 0:
    y_val_pred = (model.predict(X_val))
    val_mae = mean_absolute_error(y_val, y_val_pred)
    val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
    val_mask = (y_val != 0)
    val_mape = (
        np.mean(
            np.abs(
                (y_val[val_mask]-y_val_pred[val_mask])
                /
                y_val[val_mask]))* 100)
    print("VALIDATION PERFORMANCE - 2024")
    print(f"MAE  : {val_mae:,.2f}")
    print(f"RMSE : {val_rmse:,.2f}")
    print(f"MAPE : {val_mape:.2f}%" )

VALIDATION PERFORMANCE - 2024
MAE  : 9,547.95
RMSE : 14,220.85
MAPE : 1353.51%


In [ ]:
test["Prediction_Backprop"] = (model.predict(X_test_all))
#Passenger predictions theoretically should notbe negative. If the neural network produces a negative value, set it to zero.
test["Prediction_Backprop"] = (test["Prediction_Backprop"]
    .clip(lower=0 ))

#**TEST EVALUATION**

In [ ]:
test_evaluation = (
    test[test["Was_Missing"] == 0]
    .copy())
y_test = (test_evaluation["Passengers"])
y_pred = (test_evaluation["Prediction_Backprop"])

##test MAE, RMSE, MAPE

In [ ]:
test_mae = mean_absolute_error(y_test,y_pred)
test_rmse = np.sqrt(mean_squared_error(y_test,y_pred))
test_mask = (y_test != 0)
test_mape = (np.mean(np.abs((y_test[test_mask]-y_pred[test_mask])/y_test[test_mask]))* 100)

In [ ]:
print("BACKPROPAGATION TEST PERFORMANCE")
print("2025-2026")
print(f"MAE  : {test_mae:,.2f}")
print(f"RMSE : {test_rmse:,.2f}")
print(f"MAPE : {test_mape:.2f}%")

BACKPROPAGATION TEST PERFORMANCE
2025-2026
MAE  : 13,643.80
RMSE : 18,961.36
MAPE : 2272.97%
